# Módulo 17: Metodología — Guía del ejercicio

En este ejercicio vas a aplicar la metodología de un proyecto de ciencia de datos a un problema logístico: primero planteas el objetivo de negocio, los datos y los riesgos, y después construyes en Python una primera solución de rutas y la evalúas con un KPI medible.

- Si necesitas repasar los conceptos, consulta la lección [m17-metodologia-leccion.md](m17-metodologia-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m17-metodologia-solucion.ipynb](m17-metodologia-solucion.ipynb).

## Contexto

**Transvalores del Valle** es una empresa ficticia de transporte de valores que reabastece de efectivo los cajeros automáticos de un banco regional. Cada mañana, un camión blindado sale de la base de la empresa, visita los cajeros que el banco marcó con **alerta de efectivo bajo** y regresa a la base.

Hoy los custodios visitan los cajeros **en el orden en que llegaron las alertas**. El resultado son jornadas largas, horas extra, más combustible y más kilómetros con efectivo a bordo, que es justo cuando el camión está más expuesto. Además, el banco exige que los cajeros del distrito financiero se reabastezcan **antes de las 11:00**, porque al mediodía tienen su pico de retiros.

La gerencia de operaciones te pide plantear el proyecto con una metodología de ciencia de datos y construir una prueba de concepto con las alertas de un día típico.

## Datos

La celda inicial genera, con una semilla fija, las 15 alertas de un día en el DataFrame `cajeros` y la ubicación de la base en `base`. Las coordenadas están en kilómetros sobre un plano de la ciudad.

| Columna | Descripción |
| --- | --- |
| `cajero` | Identificador del cajero (`C01` a `C15`) |
| `orden_alerta` | Orden en que llegó la alerta de efectivo bajo (1 = la primera) |
| `zona` | Zona de la ciudad: `Norte`, `Centro` u `Oriente` |
| `x_km`, `y_km` | Coordenadas del cajero (km) |
| `servicio_min` | Minutos que tarda el reabastecimiento en el cajero |
| `antes_11h` | `True` si el cajero debe reabastecerse antes de las 11:00 |

`base` es una tupla `(x_km, y_km)` con la ubicación de la base de la empresa.

Para estimar horarios, usa estos supuestos de operación: el camión sale de la base a las **7:00**, circula a una velocidad media de **25 km/h** y la distancia entre dos puntos es la distancia en línea recta.

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta (`uv run jupyter lab` desde la raíz del repositorio).
- Las tareas 1 a 3 son preguntas escritas: respóndelas en la celda de texto que las sigue, con datos concretos del caso y conceptos de la lección.
- Las tareas 4 a 7 son de Python: escribe tu código en las celdas vacías y acompaña los resultados con una interpretación breve. Puedes añadir más celdas si lo necesitas.
- No modifiques la celda inicial: genera los datos con una semilla fija para que tus resultados sean comparables con la solución.

La celda siguiente importa las librerías y genera los datos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Tres zonas con cajeros agrupados alrededor de un centro (x, y) en km
zonas = {"Norte": ((6, 16), 5), "Centro": ((11, 9), 6), "Oriente": ((17, 4), 4)}
filas = []
for zona, ((cx, cy), n) in zonas.items():
    for x, y in rng.normal([cx, cy], 1.6, size=(n, 2)):
        filas.append({"zona": zona, "x_km": round(x, 2), "y_km": round(y, 2)})

# Las alertas llegan en un orden que no tiene relación con la ubicación
cajeros = pd.DataFrame(filas).sample(frac=1, random_state=42).reset_index(drop=True)
cajeros.insert(0, "orden_alerta", np.arange(1, len(cajeros) + 1))
cajeros.insert(0, "cajero", [f"C{i:02d}" for i in range(1, len(cajeros) + 1)])
cajeros["servicio_min"] = rng.choice([10, 15, 20, 30], size=len(cajeros), p=[0.25, 0.4, 0.25, 0.1])
cajeros["antes_11h"] = False
cajeros.loc[rng.choice(len(cajeros), 4, replace=False), "antes_11h"] = True

base = (3.0, 3.0)

print("Base:", base)
cajeros

## Tarea 1. Objetivo de negocio y KPIs

1. Redacta el objetivo de negocio del proyecto en una o dos frases. Debe decir qué se quiere mejorar y para qué, no qué algoritmo se va a usar.
2. Propón al menos cuatro KPIs medibles para saber si el proyecto funciona. Para cada uno indica cómo se calcula y en qué unidad se mide.
3. Explica qué aporta la minería de datos a este problema, más allá de calcular una ruta corta un día concreto.

*Escribe aquí tu respuesta.*

## Tarea 2. Datos necesarios y fuentes

1. Enumera los conjuntos de datos que necesitaría un sistema real de rutas para Transvalores del Valle. Para cada uno indica qué contiene y de qué fuente saldría (la empresa, el banco o un proveedor externo).
2. Señala qué limpieza o transformación necesitaría cada fuente antes de usarse.
3. Compara esa lista con los datos de este ejercicio: ¿qué información importante falta en `cajeros`?

*Escribe aquí tu respuesta.*

## Tarea 3. Riesgos de implementación y aceptación del cliente

1. Identifica al menos cuatro riesgos de implementación: técnicos, organizacionales y propios del transporte de valores.
2. Explica qué haría que la gerencia de operaciones, los custodios y el banco aceptaran la solución. Propón cómo demostrar el beneficio antes de llevarla a toda la flota.

*Escribe aquí tu respuesta.*

## Tarea 4. Matriz de distancias y ruta de referencia

1. Construye un arreglo `puntos` de 16 filas y 2 columnas: la base en la fila 0 y los cajeros en las filas 1 a 15, en el orden del DataFrame. Así, el cajero de la fila `i` de `cajeros` es el punto `i + 1`.
2. Calcula la matriz de distancias euclidianas `D`, de tamaño 16 × 16, con $D_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$. Comprueba que es simétrica y que su diagonal es cero.
3. Escribe una función `distancia_total(ruta, D)` que reciba una ruta como lista de índices de `puntos`, que empieza y termina en la base (por ejemplo, `[0, 3, 1, 2, 0]`), y devuelva la suma de las distancias de sus tramos.
4. La **ruta de referencia** es la práctica actual: visitar los cajeros en el orden de `orden_alerta`. Constrúyela y calcula su distancia total en km.

In [ ]:
# Tu código aquí

## Tarea 5. Heurística del vecino más cercano

1. Implementa la heurística del **vecino más cercano**: desde la base, ir siempre al cajero no visitado más cercano al punto actual y, cuando no queden cajeros, volver a la base.
2. Calcula la distancia total de la ruta resultante.
3. Calcula el ahorro respecto a la ruta de referencia:

$$\text{ahorro (\%)} = \frac{d_{\text{referencia}} - d_{\text{ruta}}}{d_{\text{referencia}}} \times 100$$

In [ ]:
# Tu código aquí

## Tarea 6. Mejora 2-opt y comparación del KPI

1. *(Opcional, pero recomendado.)* Mejora la ruta del vecino más cercano con **2-opt**: para cada par de posiciones `i < j` de la ruta (sin mover la base de los extremos), invierte el tramo `ruta[i:j+1]` si eso reduce la distancia total. Repite el recorrido completo hasta que ninguna inversión mejore la ruta.
2. Escribe una función que, para una ruta, estime la hora de llegada a cada cajero y la hora de regreso a la base con los supuestos de la sección Datos: salida a las 7:00, 25 km/h y, en cada cajero, `servicio_min` minutos de trabajo antes de seguir.
3. Reúne en una tabla, para cada ruta (referencia, vecino más cercano y, si la hiciste, 2-opt), la distancia total en km, el ahorro (%) respecto a la referencia, la duración de la jornada en horas y la hora de regreso a la base.
4. Interpreta la tabla: ¿cuánto se ahorra con cada método? ¿Alguna ruta permite terminar en una jornada de 8 horas?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 7. Gráfico de las rutas y límites de la heurística

1. Dibuja las rutas en una figura con un panel por ruta: la base con un marcador distinto, los cajeros coloreados por zona y los tramos de la ruta como líneas. Añade título a cada panel con su distancia total.
2. Para cada ruta, cuenta cuántos cajeros con `antes_11h == True` reciben la visita **después** de las 11:00.
3. Si hiciste 2-opt, recorre la ruta 2-opt en sentido contrario. ¿Cambia la distancia? ¿Cambia el número de cajeros atendidos tarde?
4. Explica los límites de estas heurísticas para el caso real: qué no tienen en cuenta y qué haría falta para pasar de esta prueba de concepto a un sistema en producción.

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Antes de terminar

- [ ] Las tareas 1 a 3 tienen respuestas concretas para Transvalores del Valle, no genéricas.
- [ ] El notebook se ejecuta de principio a fin sin errores.
- [ ] La matriz de distancias es simétrica y con diagonal cero.
- [ ] Todas las rutas empiezan y terminan en la base y visitan cada cajero exactamente una vez.
- [ ] La tabla de la Tarea 6 compara las rutas con el mismo KPI y el ahorro en porcentaje.
- [ ] Cada resultado va acompañado de una interpretación breve.

Cuando termines, compara tu trabajo con la solución: [m17-metodologia-solucion.ipynb](m17-metodologia-solucion.ipynb).